# Step 4c — Debate: A Society of Minds

The debate method asks: can a model do better by *arguing with itself*?
No search tool this time — debate is closed book by construction. Three
copies of the same model:

1. **Round 0** — each agent answers independently.
2. **Rounds 1–2** — each agent sees the *other* agents' answers and
   reasoning, then answers again, revised or not.
3. **Verdict** — majority vote.

This is the "society of minds" setup (Du et al. 2023). Instead of calling
a prebuilt toolkit function first, this notebook **builds a one-round
debate from scratch**, piece by piece, so the machinery is transparent —
then shows the toolkit version that runs the real thing.

The building block is the **openai-agents SDK**. Two consequences of that
choice: the SDK targets OpenAI's Responses API, so only GPT models can
debate (Gemini sits this condition out), and it is async-native — Jupyter
already runs an event loop, so we `await` directly (IPython supports
top-level `await`; plain scripts use the sync wrappers instead).

Mind the meter: a full debate costs **9 LLM calls per question**
(3 agents × 3 rounds); this notebook's from-scratch walkthrough adds 6
more.

## 1. The building block: an Agent

An `Agent` is a role (our same answering system prompt), a model, and an
`output_type` (our same `Answer` schema, which the SDK validates).
`Runner.run` executes one turn. One setup line points the SDK at the same
API key the rest of the toolkit uses:

In [1]:
from agents import Agent, Runner, set_default_openai_client, set_tracing_disabled
from openai import AsyncOpenAI

from toolkit import prompts
from toolkit.answers import Answer
from toolkit.providers import PROVIDER_ENV, load_api_key
from toolkit.utils import load_jsonl

set_default_openai_client(AsyncOpenAI(api_key=load_api_key(PROVIDER_ENV["openai"])))

# By default, the SDK automatically records and sends agent run events—such as LLM generations,
# tool calls, and handoffs—to the OpenAI Traces dashboard. Setting this to True disables this behavior.
# This is useful to eliminate background latency, prevent error logs when using alternative LLM
# providers (like Gemini, Groq, or Azure OpenAI), or stop sensitive data from being uploaded to the cloud.
set_tracing_disabled(True)

MODEL = "gpt-5.4-mini-2026-03-17"
selected = load_jsonl("../../data/questions/selected_questions.jsonl")
question = selected[0]
user_prompt = prompts.build_answer_user_prompt(
    question["question"], question["options"]
)

debater = Agent(
    name="Debater",
    model=MODEL,
    instructions=prompts.ANSWER_SYSTEM_PROMPT,
    output_type=Answer,
)

result = await Runner.run(debater, user_prompt)
result.final_output

Answer(answer_letter='B', confidence=0.9, reasoning='The AI detection review of Pope Leo XIV’s collection was conducted by Proudly Human, which is known for assessing whether text was generated by AI. The other options are not the reviewing entity.')

In [2]:
print(result.final_output.answer_letter)
print(result.final_output.confidence)
print(result.final_output.reasoning[:80])

B
0.9
The AI detection review of Pope Leo XIV’s collection was conducted by Proudly Hu


In [3]:
result.__dict__

{'input': "QUESTION:\nWhich company conducted the AI detection review of Pope Leo XIV's collection of speeches and writings, Maps of Hope?\n\nOPTIONS:\nA. Breaking News Australia\nB. Proudly Human\nC. Australian Catholic University\nD. The Vatican Publishing House\n",
 'new_items': [MessageOutputItem(agent=Agent(name='Debater', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='You are an expert news-quiz contestant. Each question was written from a\nrecently published news article (within the last few weeks). You are NOT\ngiven the article — answer from what you know or can find.\n\nRules:\n- Pick the single best option: exactly one of A, B, C, or D.\n- Always commit to one letter, even if you are unsure.\n- Give 1-2 sentences of reasoning and a confidence between 0 and 1.\n', prompt=None, handoffs=[], model='gpt-5.4-mini-2026-03-17', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None

One agent, one turn, one schema-validated `Answer` — exactly the
closed-book condition, rebuilt on the SDK.

## 2. Round 0 — three independent answers

A debate starts with independent opinions, so we run the *same* agent
three times concurrently (`asyncio.gather`). Each run is a separate API
call with its own sampling, so the three "agents" can disagree — that
disagreement is the raw material of the debate:

In [4]:
import asyncio

N_AGENTS = 3

independent_runs = await asyncio.gather(
    *(Runner.run(debater, user_prompt) for _ in range(N_AGENTS))
)
round0 = [
    {
        "agent": agent_index,
        "answer_letter": run.final_output.answer_letter,
        "confidence": run.final_output.confidence,
        "reasoning": run.final_output.reasoning,
    }
    for agent_index, run in enumerate(independent_runs)
]

for entry in round0:
    print(
        f"agent {entry['agent']}: {entry['answer_letter']} ({entry['confidence']:.2f}) {entry['reasoning'][:80]}"
    )

agent 0: B (0.91) Proudly Human is the AI-detection firm known for reviewing authorship and origin
agent 1: B (0.86) Proudly Human is the company known for conducting AI-detection reviews of texts 
agent 2: B (0.83) Proudly Human is the organization known for doing AI-authenticity/detection revi


## 3. A revision round

Now each agent gets to see the *other* agents' answers — never its own —
and answer again. The revision prompt is just the question plus a "PEER
ANSWERS" block; here is exactly what agent 0 would read:

In [5]:
peers_of_agent0 = [entry for entry in round0 if entry["agent"] != 0]
print(prompts.build_debate_revision_prompt(
    question["question"], question["options"], peers_of_agent0
))

QUESTION:
Which company conducted the AI detection review of Pope Leo XIV's collection of speeches and writings, Maps of Hope?

OPTIONS:
A. Breaking News Australia
B. Proudly Human
C. Australian Catholic University
D. The Vatican Publishing House

PEER ANSWERS:
Agent 1 answered B (confidence 0.86): Proudly Human is the company known for conducting AI-detection reviews of texts for authenticity. The other options are not known for this kind of analysis.
Agent 2 answered B (confidence 0.83): Proudly Human is the organization known for doing AI-authenticity/detection reviews of public figures’ writing and speech collections. The other options are not the reported reviewer of Maps of Hope.



The reviser is a second `Agent` with a different system prompt — same
commitment rules, but instructed to weigh the peers' arguments against
its own knowledge:

In [6]:
print(prompts.DEBATE_REVISION_SYSTEM_PROMPT)

You are an expert news-quiz contestant. Each question was written from a
recently published news article (within the last few weeks). You are NOT
given the article — answer from what you know.

Other contestants answered the same question; their answers and reasoning
are shown to you as additional advice. Weigh their arguments against your
own knowledge, then give your own (possibly revised) final answer.

Rules:
- Pick the single best option: exactly one of A, B, C, or D.
- Always commit to one letter, even if you are unsure.
- Give 1-2 sentences of reasoning and a confidence between 0 and 1.



In [7]:
reviser = Agent(
    name="Reviser",
    model=MODEL,
    instructions=prompts.DEBATE_REVISION_SYSTEM_PROMPT,
    output_type=Answer,
)

revision_turns = []
for agent_index in range(N_AGENTS):
    peer_answers = [entry for entry in round0 if entry["agent"] != agent_index]
    revision_turns.append(Runner.run(
        reviser,
        prompts.build_debate_revision_prompt(
            question["question"], question["options"], peer_answers
        ),
    ))
revised_runs = await asyncio.gather(*revision_turns)
round1 = [
    {
        "agent": agent_index,
        "answer_letter": run.final_output.answer_letter,
        "confidence": run.final_output.confidence,
        "reasoning": run.final_output.reasoning,
    }
    for agent_index, run in enumerate(revised_runs)
]

for before, after in zip(round0, round1):
    change_marker = "" if before["answer_letter"] == after["answer_letter"] else "  <- changed"
    print(f"agent {after['agent']}: {before['answer_letter']} -> "
          f"{after['answer_letter']} ({after['confidence']:.2f}){change_marker}")

agent 0: B -> B (0.90)
agent 1: B -> B (0.93)
agent 2: B -> B (0.98)


## 4. The verdict — majority vote

The final round is aggregated deterministically, so the same transcript
always grades the same way: (1) most votes wins; (2) a tie goes to the
tied letter with the highest *mean* confidence; (3) a residual tie goes
alphabetically:

In [8]:
from collections import Counter

votes = Counter(entry["answer_letter"] for entry in round1)
top_vote_count = max(votes.values())
leaders = sorted(
    letter for letter, vote_count in votes.items() if vote_count == top_vote_count
)
if len(leaders) > 1:
    def mean_confidence(letter):
        scores = [
            entry["confidence"] for entry in round1
            if entry["answer_letter"] == letter
        ]
        return sum(scores) / len(scores)
    leaders.sort(key=lambda letter: (-mean_confidence(letter), letter))
verdict = leaders[0]

print(f"votes {dict(votes)} -> verdict {verdict} "
      f"(correct: {question['correct_letter']}) -> "
      f"{'CORRECT' if verdict == question['correct_letter'] else 'WRONG'}")

votes {'B': 3} -> verdict B (correct: B) -> CORRECT


That's a complete one-round debate: independent answers → peer-informed
revision → deterministic vote. Everything else is bookkeeping.

## 5. The toolkit version

`debate_question_async()` runs the same loop with the tutorial's real
settings — `DEBATE_N_AGENTS = 3` agents and `DEBATE_N_ROUNDS = 2`
revision rounds, both in `toolkit/toolkit/config.py` — plus the pieces a
real experiment needs: per-turn retries on transient API errors, the full
transcript stored on the answer record, and the same
`_aggregate_votes()` tie-break we hand-rolled above. The final record's
confidence is the mean over the winning agents, and its reasoning comes
from the most confident winner:

In [9]:
from toolkit.debate import debate_question_async

debate = await debate_question_async(question, model=MODEL)

for round_number, round_entries in enumerate(debate["debate"]["transcript"]):
    label = "independent" if round_number == 0 else f"revision {round_number}"
    print(f"--- round {round_number} ({label}) ---")
    for entry in round_entries:
        print(f"  agent {entry['agent']}: {entry['answer_letter']} "
              f"({entry['confidence']:.2f}) {entry['reasoning'][:80]}")
print(f"\nvotes {debate['debate']['vote_counts']} -> "
      f"final {debate['answer_letter']} "
      f"({'CORRECT' if debate['is_correct'] else 'WRONG'})")

--- round 0 (independent) ---
  agent 0: B (0.78) Proudly Human is the AI-detection company identified as reviewing Pope Leo XIV’s
  agent 1: B (0.88) Proudly Human is the company known for conducting AI-detection/verification revi
  agent 2: C (0.72) Australian Catholic University is the most likely institution to have conducted 
--- round 1 (revision 1) ---
  agent 0: B (0.93) Proudly Human is the company associated with AI-detection/verification reviews, 
  agent 1: B (0.95) Proudly Human is the specialist AI-detection firm referenced as having reviewed 
  agent 2: B (0.95) Proudly Human is the AI-detection firm cited as reviewing Pope Leo XIV’s collect
--- round 2 (revision 2) ---
  agent 0: B (0.98) Proudly Human is the company identified as performing the AI detection review of
  agent 1: B (0.98) Proudly Human is the AI-detection/verification company referenced as conducting 
  agent 2: B (0.97) Proudly Human is the specialist AI-detection company that would conduct such a r

vo

## 6. The full experiment, from the command line

Debating 100 questions is 900 calls, so the tutorial debates a single
contestant model. The merge script then folds every per-run answers file
— closed book, web search, and debate — into one tidy CSV:

```bash
# 04-2: debate, one model (900 calls — 9 per question)
uv run python scripts/04-2_generate_debate_answers.py \
    --model gpt-5.4-mini-2026-03-17 --parallel

# 04-3: merge every per-run file into one tidy CSV
uv run python scripts/04-3_combine_answers.py \
    --input-dir data/answers --glob 'answers_*.jsonl'
```

## 7. The map

| This notebook | Where it lives |
|---|---|
| §1–3 agents & prompts | openai-agents SDK (`Agent`, `Runner.run`); `toolkit.prompts.DEBATE_REVISION_SYSTEM_PROMPT`, `build_debate_revision_prompt()` |
| §4 the vote | `toolkit.debate._aggregate_votes()` |
| §5 one debate | `toolkit.debate.debate_question_async()` (sync: `debate_question()`); knobs `DEBATE_N_AGENTS` / `DEBATE_N_ROUNDS` in `toolkit.config` |
| §6 at scale | `scripts/04-2_generate_debate_answers.py`; `toolkit.debate.debate_questions()`; merged by `scripts/04-3_combine_answers.py` |

---

### Next up 📊

Run the sweeps from notebooks 4a–4c, then open
[`04_answer_analysis.ipynb`](../analysis/04_answer_analysis.ipynb) for the
leaderboard: accuracy by method, what search buys, and what the debate
changed.